# 01 · Market Data, Returns and Volatility
**Week 1 of _Can a Neural Network See Market Regimes Coming?_**

Before any model, we need to understand the data. In this notebook we:
1. download daily Nifty 50 prices,
2. turn prices into **returns** (what investors actually care about),
3. measure **volatility** (how wild the returns are), and
4. build the two features used in every later notebook.

> Run cells top to bottom with **Shift + Enter**. Read the comments: they explain *why*, not just *what*.

In [ ]:
!pip -q install yfinance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams["figure.figsize"] = (11, 4)

## 1. Download prices
`^NSEI` is Yahoo Finance's ticker for the Nifty 50 index. We take daily closing prices from 2008,
so the data includes calm years *and* crashes (2008, 2020).

In [ ]:
def load_close(ticker, start="2008-01-01"):
    """Download daily closing prices from Yahoo Finance as a pandas Series."""
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    close = df["Close"]
    if isinstance(close, pd.DataFrame):   # newer yfinance returns a 1-column DataFrame
        close = close.iloc[:, 0]
    return close.dropna()

nifty = load_close("^NSEI")
print(nifty.head())
print("Number of trading days:", len(nifty))

In [ ]:
nifty.plot(title="Nifty 50 closing price")
plt.ylabel("index level")
plt.show()

## 2. Prices → returns
A price of 20,000 tells you nothing by itself. What matters is **how much it changed**.

- **Simple return:** $r_t = \frac{P_t}{P_{t-1}} - 1$ (e.g. 0.01 = +1%)
- **Log return:** $\ell_t = \ln P_t - \ln P_{t-1}$

For small moves they are almost equal. We use log returns for modelling because they **add up over time**:
the log return over a week is the sum of the daily log returns.

In [ ]:
df = pd.DataFrame({"close": nifty})
df["simple_ret"] = df["close"].pct_change()
df["log_ret"] = np.log(df["close"]).diff()
df = df.dropna()

print(df[["simple_ret", "log_ret"]].describe())

In [ ]:
# Are returns "normal" (bell-shaped)? Compare the histogram with a normal curve.
r = df["log_ret"]
plt.hist(r, bins=150, density=True, alpha=0.6, label="actual daily returns")
x = np.linspace(r.min(), r.max(), 400)
normal = np.exp(-(x - r.mean())**2 / (2 * r.std()**2)) / (r.std() * np.sqrt(2 * np.pi))
plt.plot(x, normal, "r", label="normal distribution, same mean & std")
plt.xlim(-0.06, 0.06); plt.legend(); plt.title("Daily log returns")
plt.show()

print("Worst 5 days:")
print(r.nsmallest(5))

**What to notice:** the real histogram is taller in the middle and has **fatter tails** than the normal curve.
Huge crash days happen far more often than a normal distribution predicts. Markets switch between quiet and
violent periods, and that is exactly the idea behind *regimes*.

## 3. Volatility
Volatility = standard deviation of returns over a recent window. We use a **20-day rolling window**
(about one trading month) and multiply by $\sqrt{252}$ to express it per year (252 trading days in a year).

In [ ]:
df["vol_20d"] = df["log_ret"].rolling(20).std() * np.sqrt(252)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
df["close"].plot(ax=ax[0], title="Nifty 50")
df["vol_20d"].plot(ax=ax[1], color="darkred", title="20-day annualised volatility")
plt.tight_layout(); plt.show()

**What to notice:** volatility comes in **clusters**. It spikes in 2008 and March 2020 and stays low for long stretches
in between. Calm periods tend to stay calm; crises tend to stay crises for a while. An HMM (next notebook) is built
to capture exactly this.

## 4. Summary statistics

In [ ]:
years = len(df) / 252
total_growth = df["close"].iloc[-1] / df["close"].iloc[0]
cagr = total_growth ** (1 / years) - 1
ann_vol = df["log_ret"].std() * np.sqrt(252)

running_peak = df["close"].cummax()
drawdown = df["close"] / running_peak - 1          # how far below the previous peak we are

print(f"Period:            {df.index[0].date()} to {df.index[-1].date()}")
print(f"CAGR:              {cagr:.2%}  (average yearly growth)")
print(f"Annual volatility: {ann_vol:.2%}")
print(f"Max drawdown:      {drawdown.min():.2%}  on {drawdown.idxmin().date()}")

drawdown.plot(title="Drawdown from previous peak", color="gray")
plt.show()

## 5. The features we will use from now on
Every later notebook uses this exact function: **log return** and **20-day volatility** for each day.

In [ ]:
def make_features(close, vol_window=20):
    """Daily log return + rolling annualised volatility."""
    df = pd.DataFrame({"close": close})
    df["ret"] = np.log(df["close"]).diff()                                   # log return
    df["vol"] = df["ret"].rolling(vol_window).std() * np.sqrt(252)           # annualised volatility
    return df.dropna()

features = make_features(nifty)
features.tail()

## Check your understanding
1. Why do we model returns instead of prices?
2. Why are log returns convenient? What is the log return over 5 days in terms of daily log returns?
3. What does "volatility clustering" mean? Where do you see it in the plots?
4. Why multiply daily volatility by √252?
5. What is a drawdown, and why do investors care about the *maximum* drawdown?